In [19]:
from openai import OpenAI
import instructor
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient


In [20]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

In [21]:
prompt = """
You are a helpful assistant that can answer questions and help with tasks.
Return the answer to the question
Question: what is your name?
"""

In [22]:
response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[{"role": "system", "content": prompt}],
    reasoning_effort="none"
)
print(response.choices[0].message.content)

My name is **ChatGPT**.


### Add Instructor - for structured outputs

In [8]:
client = instructor.from_provider("openai/gpt-5.4-nano", mode=instructor.Mode.RESPONSES_TOOLS)

class Answer(BaseModel):
    answer: str = Field(description="The answer to the question")


In [9]:
response = client.create(messages=[{"role": "system", "content": prompt}], 
reasoning={"effort": "none"},
response_model=Answer
)

In [10]:
response

Answer(answer='I’m ChatGPT.')

In [ ]:
response, raw_response = client.create_with_completion(messages=[{"role": "system", "content": prompt}], 
reasoning={"effort": "none"},
response_model=Answer
)

response

Answer(answer='I’m ChatGPT.')

In [12]:
raw_response

Response(id='resp_000a44af23f8a287006a4a3b04f86481a3bca5568adf9937e4', created_at=1783249668.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"I’m ChatGPT."}', call_id='call_OlnjcTxv1I32P49FULMOL9fl', name='Answer', type='function_call', id='fc_000a44af23f8a287006a4a3b059dd481a3bd6164f7a298782b', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'The answer to the question', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Correctly extracted `Answer` with all the required parameters with correct types')], top_p=0.98, background=False, 

In [13]:
class AnswerWithReasoning(BaseModel):
    reasoning: str = Field(description="The reasoning for the answer")
    answer: str = Field(description="The answer to the question")
    

In [14]:
response, raw_response = client.create_with_completion(messages=[{"role": "system", "content": prompt}], 
reasoning={"effort": "none"},
response_model=AnswerWithReasoning
)

response

AnswerWithReasoning(reasoning='The user asks for my name; as an AI assistant, I should provide a brief identification.', answer='You can call me ChatGPT.')

## RAG Pipeline

In [27]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    
    

In [28]:
model = "text-embedding-3-small"
collection_name="Amazon-shopping-collection-01"
qdrant_client = QdrantClient(url="http://localhost:6333")
model_name = "gpt-5.4-nano"
openai = OpenAI()

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

def retrieve_context(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(collection_name=collection_name, query=query_embedding, limit=k)
    context_ids = []
    context = []
    scores = []
    ratings = []

    print(results)
    for pt in results.points:
        context_ids.append(pt.payload['parent_asin'])
        context.append(pt.payload['preprocessed_description'])
        scores.append(pt.score)
        ratings.append(pt.payload['average_rating'])
        
    return {
        'retrieved_context_ids': context_ids,
        'retrieved_context': context,
        'similarity_scores': scores,
        'retrieved_context_ratings': ratings
    }

def process_context(context):
    formated_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
        formated_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formated_context


def build_prompt(context, query):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - answer the question based on the provided context only.
    - never use word context and refer to it as the available products.
    - do not use markdown formatting.

    Context:
    {context}

    Question:
    {query}
    """

    return prompt

def generate_answer(prompt):
    
    #response = openai.chat.completions.create(
    #        model=model_name,
    #        messages=[{"role":"system","content": prompt}]
    #                 )
    response, raw_response = client.create_with_completion(messages=[{"role":"system","content": prompt}], 
    reasoning={"effort": "none"},
    response_model=RAGGenerationResponse
    )
    return response


def rag_pipeline(query, top_k= 5):
    retrieved_context = retrieve_context(query, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    llm_answer = generate_answer(prompt)
    
    final_answer= {
        "query": query,
        "data_object": llm_answer,
        "answer": llm_answer.answer,
        "retrieved_context_ids": retrieved_context['retrieved_context_ids'],
        "retrieved_context": retrieved_context['retrieved_context'],
    }
    return final_answer

In [36]:
rag_pipeline("do you have bluetooth earbuds?")

points=[ScoredPoint(id=43, version=1, score=0.45706463, payload={'preprocessed_description': 'Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds Lightweight and Noise Canceling Wireless Earbuds Fit for Workout with Built-in Magnet, Green[Incredible Sound Quality] – APTX codec offers premium audio sound quality. Bluetooth 4.1 technology enables quick transmission and seamless music streaming. These headphones are compatible with most Bluetooth-enabled devices. [Built-in Magnets & Lightweight] – With built-in magnets, you can easily hang your earbuds around your neck. Weighing only 0.56 oz. net weight, these earbuds are a great fit for athletes, moms, and students. [Long-lasting Battery & Super-Comfortable Wearing Experience] – Up to 8 hours talk or music time, or 175 hours stand-by time with 2 hours charging. ATGOIN Bluetooth headphones is the perfect accessory to all your music needs. [CVC Noise Canceling & Sweat Resistant] – With CVC no

{'query': 'do you have bluetooth earbuds?',
 'data_object': RAGGenerationResponse(answer='Yes. We have Bluetooth earbuds available: ATGOIN Sweatproof Wireless Earbuds (green) and you may also find them as compatible with Bluetooth devices.'),
 'answer': 'Yes. We have Bluetooth earbuds available: ATGOIN Sweatproof Wireless Earbuds (green) and you may also find them as compatible with Bluetooth devices.',
 'retrieved_context_ids': ['B07K6LD7D5',
  'B01K6P08FA',
  'B075NR47FN',
  'B086QGSXRB',
  'B085ZTXZFV'],
 'retrieved_context': ['Bluetooth Headphones, Wireless Headphones, ATGOIN Sweatproof High Fidelity Stereo Bluetooth Earbuds Lightweight and Noise Canceling Wireless Earbuds Fit for Workout with Built-in Magnet, Green[Incredible Sound Quality] – APTX codec offers premium audio sound quality. Bluetooth 4.1 technology enables quick transmission and seamless music streaming. These headphones are compatible with most Bluetooth-enabled devices. [Built-in Magnets & Lightweight] – With buil